In [14]:
import pandas as pd
import numpy as np
import ast
import json

# 1. CARICAMENTO
# Carichiamo il file pulito dalla Fase 1
df = pd.read_csv('../data/processed/KICKSTARTER_CLEAN_BASE.csv', low_memory=False)
print(f"📊 Dataset iniziale: {df.shape}")

# 2. TARGET (0/1)
# Trasformiamo la colonna target in numeri
if 'state' in df.columns:
    df['target'] = df['state'].apply(lambda x: 1 if str(x).lower().strip() == 'successful' else 0)

# 3. FEATURE ENGINEERING: DATE
# Convertiamo in datetime
if 'launched_at' in df.columns:
    df['launched_at'] = pd.to_datetime(df['launched_at'], unit='s')
if 'deadline' in df.columns:
    df['deadline'] = pd.to_datetime(df['deadline'], unit='s')
if 'created_at' in df.columns:
    df['created_at'] = pd.to_datetime(df['created_at'], unit='s')

# Creiamo nuove feature
df['duration_days'] = (df['deadline'] - df['launched_at']).dt.days
df['launch_month'] = df['launched_at'].dt.month
df['launch_day'] = df['launched_at'].dt.dayofweek # 0=Lun, 6=Dom

# Giorni di preparazione (se disponibili)
if 'created_at' in df.columns:
    df['preparation_days'] = (df['launched_at'] - df['created_at']).dt.days

# 4. FEATURE ENGINEERING: TESTO
# Riempiamo i nulli con stringa vuota per evitare errori
df['name'] = df['name'].fillna('')
df['blurb'] = df['blurb'].fillna('')

# Lunghezza
df['name_len'] = df['name'].astype(str).apply(len)
df['blurb_len'] = df['blurb'].astype(str).apply(len)

# Feature "Clickbait": ha il punto interrogativo? (CORRETTO QUI)
# Usiamo regex=False perché cerchiamo solo il carattere '?', è più veloce e sicuro
df['has_question'] = df['name'].astype(str).str.contains('?', regex=False).astype(int)

# 5. ESTRAZIONE CATEGORIA
def extract_category(x):
    try:
        if isinstance(x, str) and '{' in x:
            # Safe parsing
            return ast.literal_eval(x).get('slug', '').split('/')[0]
        return x
    except:
        return 'Unknown'

if 'category' in df.columns:
    df['main_category'] = df['category'].apply(extract_category)

print("✅ Feature Engineering completato.")
if 'duration_days' in df.columns:
    print(df[['duration_days', 'name_len', 'main_category']].head())

📊 Dataset iniziale: (188954, 43)
✅ Feature Engineering completato.
   duration_days  name_len main_category
0             60        57        comics
1             30        18         music
2             45        43        design
3              7        47         games
4             28         4           art


In [15]:
# 1. GESTIONE OUTLIER (GOAL)
# Come visto nell'istogramma, applichiamo il logaritmo per "schiacciare" i valori estremi
df['goal'] = pd.to_numeric(df['goal'], errors='coerce')
df['goal_log'] = np.log1p(df['goal']) # Logaritmo naturale (log(x+1))

# 2. GESTIONE MISSING VALUES
# Strategia:
# - Numerici: Mediana (più robusta agli outlier della media)
# - Categorici: "Unknown"
cols_num = ['duration_days', 'preparation_days', 'name_len', 'blurb_len']
for col in cols_num:
    if col in df.columns:
        mediana = df[col].median()
        df[col] = df[col].fillna(mediana)

# Categorie mancanti
df['country'] = df['country'].fillna('Unknown')
df['main_category'] = df['main_category'].fillna('Unknown')

print("✅ Pulizia completata (Log Goal + Imputazione Missing).")

✅ Pulizia completata (Log Goal + Imputazione Missing).


In [16]:
# Lista colonne da rimuovere (come da tuo piano Fase 3)
leakage_cols = [
    'pledged', 'backers_count', 'usd_pledged', 'converted_pledged_amount', # Risultati futuri
    'state', 'state_changed_at', 'staff_pick', 'spotlight', 'percent_funded', # Spoiler
    'id', 'creator', 'profile', 'photo', 'urls', 'source_url', 'slug', 'location', # Metadata sporchi
    'category', 'launched_at', 'deadline', 'created_at', 'goal', # Ridondanti (sostituite dalle feature processate)
    'currency_symbol', 'static_usd_rate', 'currency_trailing_code' # Inutili
]

# Rimuoviamo solo quelle presenti
cols_to_drop = [c for c in leakage_cols if c in df.columns]
df_model = df.drop(columns=cols_to_drop)

print(f"✅ Leakage rimosso. Colonne rimaste: {len(df_model.columns)}")
print(df_model.columns.tolist())

✅ Leakage rimosso. Colonne rimaste: 28
['blurb', 'country', 'country_displayable_name', 'currency', 'current_currency', 'disable_communication', 'fx_rate', 'is_disliked', 'is_in_post_campaign_pledging_phase', 'is_launched', 'is_liked', 'is_starrable', 'name', 'prelaunch_activated', 'usd_exchange_rate', 'usd_type', 'video', 'sort_date', 'target', 'duration_days', 'launch_month', 'launch_day', 'preparation_days', 'name_len', 'blurb_len', 'has_question', 'main_category', 'goal_log']


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. CLEANING BASE TESTO
# Assicuriamoci che siano stringhe pulite
df_model['name'] = df_model['name'].astype(str).fillna('')

# 2. NLP AVANZATO (TF-IDF)
# Prendiamo le 50 parole più importanti dai titoli dei progetti
print("⏳ Estrazione parole chiave (NLP)...")
tfidf = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_model['name'])

# Creiamo un DataFrame con queste nuove 50 colonne numeriche
# I nomi delle colonne saranno tipo "word_art", "word_game", ecc.
tfidf_cols = [f'word_{word}' for word in tfidf.get_feature_names_out()]
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_cols, index=df_model.index)

# Uniamo le nuove colonne al nostro dataset
df_model = pd.concat([df_model, df_tfidf], axis=1)
print(f"✅ Aggiunte {len(tfidf_cols)} feature testuali (parole chiave).")

# 3. RIMOZIONE TESTO ORIGINALE
# Ora che abbiamo estratto i numeri ("word_smart"=1), possiamo buttare il testo originale
text_cols_to_drop = ['name', 'blurb', 'slug', 'currency_symbol', 'currency_trailing_code']
df_model = df_model.drop(columns=[c for c in text_cols_to_drop if c in df_model.columns])

# 4. IDENTIFICHIAMO LE CATEGORIE RIMASTE
cat_cols = ['country', 'currency', 'main_category']
cat_cols = [c for c in cat_cols if c in df_model.columns]

# 5. ENCODING FINALE
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Conversione finale per sicurezza
df_encoded = df_encoded.apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"✅ Encoding completato. Dataset finale: {df_encoded.shape}")
df_encoded.head()

⏳ Estrazione parole chiave (NLP)...
✅ Aggiunte 50 feature testuali (parole chiave).
✅ Encoding completato. Dataset finale: (188954, 125)


,country_displayable_name,current_currency,disable_communication,fx_rate,is_disliked,is_in_post_campaign_pledging_phase,is_launched,is_liked,is_starrable,prelaunch_activated,...,main_category_fashion,main_category_film & video,main_category_food,main_category_games,main_category_journalism,main_category_music,main_category_photography,main_category_publishing,main_category_technology,main_category_theater
0,0.0,0.0,False,1.000000,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,0.0,0.0,False,1.000000,False,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,0.0,0.0,False,1.000000,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,0.0,0.0,False,1.000000,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,False,False
4,0.0,0.0,False,1.339463,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [18]:
output_path = '../data/processed/KICKSTARTER_ENCODED.csv'
df_encoded.to_csv(output_path, index=False)
print(f"💾 File pronto per il training salvato in: {output_path}")

💾 File pronto per il training salvato in: ../data/processed/KICKSTARTER_ENCODED.csv
